<a href="https://colab.research.google.com/github/aqrlouhanjoauhan/PakePlus-Android-v2.1.5/blob/main/youtube_subtitle_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 YouTube Knowledge Base Cloud Extraction Engine { display-mode: "form" }
#@markdown 💡 **Click the Run button (▶) on the left to start downloading subtitles!**

import sys, os, re, shutil, json, subprocess, glob, urllib.parse
from google.colab import output, files
from IPython.display import display, HTML

# 默认测试频道（已移入注释备用）：
# DEFAULT_TEST_URL = "https://www.youtube.com/@OlderBrother_"

TARGET_URL = ""

# ==============================================================
# 1. 运行瞬间弹出原生 Prompt 引导用户粘贴
# ==============================================================
js_prompt_code = """
(async () => {
    let clipboardText = "";
    try {
        const text = await navigator.clipboard.readText();
        if (text && (text.includes('youtube.com') || text.includes('youtu.be'))) {
            clipboardText = text;
        }
    } catch(e) {}

    // 弹出提示框，优先自动读取剪贴板的油管链接
    const userEntered = prompt("🎯 Please paste your YouTube Channel or Video URL below (Ctrl+V / Cmd+V):", clipboardText);
    return { target_url: userEntered };
})();
"""

try:
    res = output.eval_js(js_prompt_code)
    if res and res.get('target_url') and res['target_url'].strip():
        TARGET_URL = res['target_url'].strip()
    else:
        TARGET_URL = ""
except Exception as e:
    TARGET_URL = ""

# ==============================================================
# 2. 空值/取消判定：如果为空或点了 Cancel，终止程序
# ==============================================================
if not TARGET_URL:
    print("\n❌ Extraction Terminated: YouTube URL is empty or process was cancelled.")
    print("💡 Please click Run (▶) again and paste a valid YouTube link.")
    sys.exit(0) # 安全退出，不继续往下执行

print(f"\n🎯 Target URL Confirmed: {TARGET_URL}")
print("⏳ Preparing cloud environment (takes ~10 seconds on first run)...")
os.system("pip install -q youtube-transcript-api yt-dlp")
from youtube_transcript_api import YouTubeTranscriptApi

base_dir = "./transcripts_temp"
if os.path.exists(base_dir):
    shutil.rmtree(base_dir)
os.makedirs(base_dir, exist_ok=True)

# Subtitle cleaner
def clean_subtitle_text(vtt_text):
    lines = vtt_text.splitlines()
    cleaned = []
    for line in lines:
        line = line.strip()
        if not line or "WEBVTT" in line or "Kind:" in line or "Language:" in line or "-->" in line:
            continue
        if re.match(r'^\d+$', line):
            continue
        line = re.sub(r'<[^>]+>', '', line)
        if not cleaned or cleaned[-1] != line:
            cleaned.append(line)
    return " ".join(cleaned)

# ==============================================================
# 3. Single Video vs Channel Detection
# ==============================================================
is_single_video = any(kw in TARGET_URL for kw in ["watch?v=", "youtu.be/", "/shorts/"])
video_list = []
channel_title = "youtube_subtitles"

if is_single_video:
    print(f"\n🎬 Mode Detected: 【Single Video】")
    cmd = ["yt-dlp", "--dump-json", "--no-playlist", TARGET_URL]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        data = json.loads(res.stdout)
        video_list.append({'id': data['id'], 'title': data.get('title', data['id'])})
        safe_title = re.sub(r'[^\w\-_]', '_', data.get('title', 'video'))[:30]
        channel_title = f"Video_{safe_title}"
else:
    print(f"\n📺 Mode Detected: 【Channel / Playlist Full Batch Download】")
    fetch_url = TARGET_URL.rstrip('/')
    if not any(fetch_url.endswith(sub) for sub in ['/videos', '/shorts', '/playlists']):
        fetch_url += '/videos'

    print(f"🎯 Target URL: {fetch_url}")

    cmd = ["yt-dlp", "--flat-playlist", "--dump-single-json", fetch_url]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        data = json.loads(res.stdout)
        channel_title = re.sub(r'[^\w\-_]', '_', data.get('title', 'Channel'))
        for entry in data.get('entries', []):
            if entry.get('id') and entry.get('_type') != 'playlist':
                video_list.append({'id': entry['id'], 'title': entry.get('title', entry['id'])})

        total_found = len(video_list)
        print(f"✅ Successfully scanned ALL {total_found} real videos!")

# ==============================================================
# 4. Extract Transcripts & Download Zip
# ==============================================================
BATCH_SIZE = 50

if not video_list:
    print("❌ Extraction Failed: Could not recognize video info. Please check the URL.")
else:
    print("\n📝 Extracting transcripts...")
    success_count = 0
    created_zips = []

    for idx, item in enumerate(video_list, 1):
        vid = item['id']
        title = item['title']

        batch_num = ((idx - 1) // BATCH_SIZE) + 1
        batch_dir = os.path.join(base_dir, f"part_{batch_num}")
        os.makedirs(batch_dir, exist_ok=True)

        safe_title = re.sub(r'[^\w\-_]', '_', title)[:40]
        filepath = os.path.join(batch_dir, f"{idx:03d}_[{vid}]_{safe_title}.txt")
        extracted_text = ""

        try:
            v_url = f"https://www.youtube.com/watch?v={vid}"
            temp_vtt_prefix = f"/tmp/sub_{vid}"

            dl_cmd = [
                "yt-dlp", "--skip-download",
                "--write-sub", "--write-auto-sub",
                "--sub-langs", "en.*,zh.*,zh-Hans,zh-Hant,all",
                "--output", f"{temp_vtt_prefix}.%(ext)s", v_url
            ]
            subprocess.run(dl_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            downloaded_files = glob.glob(f"{temp_vtt_prefix}*")
            if downloaded_files:
                selected_file = downloaded_files[0]
                for df in downloaded_files:
                    if any(lang in df.lower() for lang in ['en', 'zh', 'chinese']):
                        selected_file = df
                        break

                with open(selected_file, "r", encoding="utf-8", errors="ignore") as vf:
                    raw_vtt = vf.read()
                    extracted_text = clean_subtitle_text(raw_vtt)

                for df in downloaded_files:
                    try: os.remove(df)
                    except: pass
        except Exception:
            pass

        if not extracted_text:
            try:
                t_list = YouTubeTranscriptApi.list_transcripts(vid)
                t_obj = next(iter(t_list))
                extracted_text = " ".join([e['text'] for e in t_obj.fetch()])
            except Exception:
                pass

        if extracted_text.strip():
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(extracted_text.strip())
            success_count += 1
            print(f"  └─ [{idx}/{len(video_list)}] ✅ Success: {title[:25]}...")
        else:
            print(f"  └─ [{idx}/{len(video_list)}] ⚠️ Skipped: {title[:25]}...")

    if success_count > 0:
        print(f"\n📦 Packing finished! Successfully fetched {success_count} transcripts.")
        part_folders = [f for f in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, f))]
        part_folders.sort()

        for folder in part_folders:
            folder_path = os.path.join(base_dir, folder)
            if os.listdir(folder_path):
                zip_name = f"{channel_title}_subtitles" if len(part_folders) == 1 else f"{channel_title}_subtitles_{folder}"
                zip_filepath = shutil.make_archive(zip_name, 'zip', folder_path)
                created_zips.append(zip_filepath)

        print(f"🚀 Triggering download...")
        for zip_file in created_zips:
            files.download(zip_file)
    else:
        print("\n❌ Extraction Failed: Selected video(s) contain no valid transcripts.")